# Ant-2K analysis — MATLAB port

Ports `regions_Ant2K.m`, `Mapper_region_colour.m` and `Analyse_PCA_Antarctica.m`.

Backed by the `antarctic` package in this repo. Run top to bottom the first time —
later cells depend on `cube`, `slope_e`, `cores` and `r2` from earlier ones.

**Inputs actually used** (not the ones in the original delivery):
- ERA5 **2 m** temperature — `~/DataFiles/ae3baa6a…nc`. The file in `Mathieu/Data`
  is a 1000 hPa pressure-level download and cannot reproduce the published figure.
- **BedMachine Antarctica v3** (500 m) for surface elevation, replacing `bedmap2_interp`.

In [57]:
import sys, warnings
sys.path.insert(0, '/Users/advik/ice-top')
warnings.filterwarnings('ignore')

%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from antarctic.config import REGION_NAMES, DATA_DIR
from antarctic.elevation import default_elevation
from antarctic.regions import regions_ant2k, regions_at
from antarctic.datasets import load_era5, load_bromwich, load_ice_cores, load_aws
from antarctic.analysis import trend_1d, trend_map, merge_neighbours, pca_fields
from antarctic.helpers import ann_block_ave
from antarctic.maps import (PLATE, polar_axes, add_coast, surfm, scatterm,
                            cmocean_like, region_cmap, region_legend)

plt.rcParams['figure.dpi'] = 110

elev = default_elevation()
print('DEM :', elev.name)
print('ERA5:', load_era5().attrs['note'])

DEM : Bromwich WRF HGT (60 km)
ERA5: ERA5 t at 1000 hPa (not 2 m); latitude limited to 55S-90S


## 1. Elevation backend

BedMachine v3 at 1 km, validated against known station elevations.

In [58]:
known = {'South Pole': (-89.99, 0.0, 2835), 'Dome C': (-75.10, 123.35, 3233),
         'WAIS Divide': (-79.47, -112.09, 1766), 'Vostok': (-78.47, 106.84, 3488),
         'Dome A': (-80.37, 77.37, 4093), 'Byrd': (-80.01, -119.32, 1530),
         'Talos Dome': (-72.78, 159.07, 2315), 'Law Dome': (-66.77, 112.81, 1370)}

pd.DataFrame([{'site': k, 'interpolated_m': round(float(elev(la, lo)), 1),
               'reference_m': ref, 'diff_m': round(float(elev(la, lo)) - ref, 1)}
              for k, (la, lo, ref) in known.items()]).set_index('site')

,interpolated_m,reference_m,diff_m
site,,,
South Pole,2832.5,2835,-2.5
Dome C,3217.9,3233,-15.1
WAIS Divide,1766.8,1766,0.8
Vostok,3456.1,3488,-31.9
Dome A,3971.5,4093,-121.5
Byrd,1520.2,1530,-9.8
Talos Dome,2155.3,2315,-159.7
Law Dome,742.3,1370,-627.7


## 2. Rebuild the missing Victoria Land boundary

`LatitudeLimofVictoriaLand.txt` was never delivered. It is reconstructed by tracing
the east edge of the contiguous 2000 m surface — inside 145–190°E the plateau rule
in the MATLAB carries no elevation test of its own, so this table *is* the 2000 m
contour there.

In [59]:
from scripts.make_victoria_limit import (plateau_edge_longitude, trim_north_taper,
                                         fill_north, smooth, LAT_STEP,
                                         SMOOTH_WINDOW, LON_MIN, LON_MAX)

lats_v = np.arange(-89.75, -59.99, LAT_STEP)
raw = np.array([plateau_edge_longitude(elev, la) for la in lats_v])
lim = np.clip(smooth(fill_north(trim_north_taper(raw)), SMOOTH_WINDOW), LON_MIN, LON_MAX)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(raw, lats_v, '.', ms=4, alpha=.5, label='raw 2000 m edge')
ax.plot(lim, lats_v, '-', lw=2, label='trimmed + smoothed')
ax.plot(159.07, -72.78, 'r*', ms=14, label='Talos Dome')
ax.axvline(160, color='k', ls=':', lw=1, label="MATLAB's hard-coded 160°")
ax.set_xlabel('longitude limit (°E)'); ax.set_ylabel('latitude')
ax.set_title('Plateau / Victoria Land boundary'); ax.legend(); ax.grid(alpha=.3)
plt.show()

print(f'northern extrapolated value: {lim[-1]:.2f}°E')

ImportError: cannot import name 'trim_north_taper' from 'scripts.make_victoria_limit' (/Users/advik/ice-top/scripts/make_victoria_limit.py)

## 3. The 7 Ant-2K regions, and the region-0 bug

Lines 47–50 of `regions_Ant2K.m` assign `0` — not a region in 1..7 — and nothing
later overwrites it, because every subsequent DML/Weddell rule requires `elev < 2000`.
`caxis([1 7])` clamps 0 to the plateau colour, which is why it is invisible in the
original figure.

In [ ]:
lat_g = np.arange(-90.0, -59.95, 0.1)
lon_g = np.arange(0.0, 360.05, 0.1)
LA, LO = np.meshgrid(lat_g, lon_g, indexing='ij')
Z = elev(LA, LO)

ri_raw = regions_ant2k(LA, LO, Z, edml_zero=True)    # faithful to the MATLAB
ri_fix = regions_ant2k(LA, LO, Z, edml_zero=False)   # corrected
cmap_r, norm_r = region_cmap()

fig = plt.figure(figsize=(15, 7.5))
_, ax1 = polar_axes(fig, (1, 2, 1), lat_max=-61.0)
surfm(ax1, lat_g, lon_g, np.where(ri_raw == 0, 1.0, ri_raw), cmap=cmap_r, norm=norm_r)
add_coast(ax1); ax1.set_title('As drawn by the MATLAB')

_, ax2 = polar_axes(fig, (1, 2, 2), lat_max=-61.0, labels=False)
surfm(ax2, lat_g, lon_g, ri_fix, cmap=cmap_r, norm=norm_r)
ax2.contourf(lon_g, lat_g, np.where(ri_raw == 0, 1.0, np.nan), levels=[.5, 1.5],
             colors='none', hatches=['////'], transform=PLATE, zorder=4)
add_coast(ax2)
n0 = int(np.sum((ri_raw == 0) & (Z > 10)))
ax2.set_title(f'Hatched: {n0:,} cells labelled 0, not 1')
region_legend(ax2)
plt.show()

print(f'region-0 land cells: {n0:,}  ({n0 / np.sum((ri_fix == 1) & (Z > 10)):.0%} of the plateau)')

### Region assignment at named sites

In [ ]:
sites = {'Dome C': (-75.10, 123.35), 'Vostok': (-78.47, 106.84),
         'South Pole': (-89.99, 0.0), 'EDML/Kohnen': (-75.00, 0.07),
         'Talos Dome': (-72.78, 159.07), 'Byrd': (-80.01, -119.32),
         'Law Dome': (-66.77, 112.81), 'James Ross Is': (-64.20, -57.68),
         'Berkner Is': (-79.55, -45.68), 'Taylor Dome': (-77.78, 158.72),
         'Neumayer': (-70.65, -8.25), "Dumont d'Urville": (-66.66, 140.00)}

pd.DataFrame([{'site': n, 'matlab': regions_at(la, lo, edml_zero=True)[0],
               'corrected': regions_at(la, lo, edml_zero=False)[0],
               'region': REGION_NAMES.get(regions_at(la, lo, edml_zero=False)[0], '?')}
              for n, (la, lo) in sites.items()]).set_index('site')

## 4. ERA5 source check

Confirms which file is loaded. With the correct 2 m file these errors are small and
uniform; with the 1000 hPa file in `Mathieu/Data` the plateau sites were 25–33 °C
too warm.

In [ ]:
era5 = load_era5()
print(era5.attrs['note'])
print(era5.attrs['source_file'])
m = era5.mean('time')

check = {"Dumont d'Urville": (-66.7, 140.0, -11), 'Halley': (-75.35, 333.6, -18),
         'South Pole': (-90, 0, -49), 'Vostok': (-78.5, 106.9, -55),
         'Dome C': (-75.1, 123.35, -54)}
pd.DataFrame([{'site': k,
               'file_degC': round(float(m.sel(lat=la, lon=lo, method='nearest')), 1),
               'true_annual_mean': tru,
               'error': round(float(m.sel(lat=la, lon=lo, method='nearest')) - tru, 1)}
              for k, (la, lo, tru) in check.items()]).set_index('site')

## 5. Annual means and the mean temperature map

In [ ]:
ann = era5.groupby('time.year').mean('time')
years = ann.year.values.astype(float)
lat_e, lon_e = ann.lat.values, ann.lon.values
cube = ann.values.astype('float32')
print(cube.shape, f'{years[0]:.0f}-{years[-1]:.0f}, NaN fraction {np.isnan(cube).mean():.3f}')

fig = plt.figure(figsize=(7.5, 7))
_, ax = polar_axes(fig, 111, lat_max=-62.0)
mesh = surfm(ax, lat_e, lon_e, np.nanmean(cube, axis=0),
             cmap=cmocean_like('balance', 20), vmin=-60, vmax=0)
add_coast(ax); fig.colorbar(mesh, ax=ax, shrink=.7, label='Temperature (°C)')
ax.set_title(f'ERA5 mean {years[0]:.0f}-{years[-1]:.0f}')
plt.show()

## 6. AWS station trends, 1980–2020

In [ ]:
YEAR_START, YEAR_STOP, TREND_LIM = 1980, 2020, 0.65
LEV = np.arange(-0.65, 0.66, 0.1)              # the published figure's levels
cmap_t = cmocean_like('balance', 13)
stations = load_aws()

aws = []
for st in stations:
    slope, p, n = trend_1d(st.year, st.temp, YEAR_START, YEAR_STOP)
    aws.append({'station': st.name, 'lat': st.lat, 'lon': st.lon,
                'trend_C_per_dec': round(10 * slope, 3), 'p': round(p, 3), 'n': n})
aws = pd.DataFrame(aws).set_index('station')
aws['sig'] = aws.p < 0.05
aws.sort_values('trend_C_per_dec', ascending=False)

## 7. Gridded ERA5 trends

In [ ]:
slope_e, pval_e = trend_map(years, cube, YEAR_START, YEAR_STOP)

# what the datenum/365 axis bug in the MATLAB actually cost
slope_bug, _ = trend_map(years + 1.5, cube, YEAR_START, YEAR_STOP)
print(f'mean |trend difference| from the /365 axis bug: '
      f'{np.nanmean(np.abs(10 * (slope_e - slope_bug))):.4f} °C/dec')

fig = plt.figure(figsize=(8, 7.5))
_, ax = polar_axes(fig, 111, lat_max=-62.0)
mesh = ax.contourf(lon_e, lat_e, 10 * slope_e, levels=LEV, cmap=cmap_t,
                   extend='both', transform=PLATE)
scatterm(ax, aws.lat, aws.lon, s=170, c=aws.trend_C_per_dec, cmap=cmap_t,
         vmin=-TREND_LIM, vmax=TREND_LIM, edgecolors='0.1', linewidths=2.5)
add_coast(ax)
fig.colorbar(mesh, ax=ax, shrink=.65, label='Trend (°C/dec)', ticks=LEV[::2])
ax.set_title(f'Temperature trends ERA5 {YEAR_START}–{YEAR_STOP}')
plt.show()

## 8. Ice-core δ¹⁸O trends (2× scaling)

Defines `cores`, `slope_c`, `pval_c` — needed by the next two cells.

In [ ]:
cores = load_ice_cores()
CORE_START, CORE_STOP = 1950, 2020
print(f'{len(cores)} usable cores (1 empty placeholder dropped)')

end = np.array([a.max() for a in cores.age])
print(f'records end between {end.min():.0f} and {end.max():.0f} — '
      f'so "{CORE_START}–{CORE_STOP}" is really to {end.max():.0f} at best')

s2 = np.array([trend_1d(a, d * 2.0, CORE_START, CORE_STOP)[:2]
               for a, d in zip(cores.age, cores.d18O)])
slope_c, pval_c = s2[:, 0], s2[:, 1]
sig_c = np.isfinite(pval_c) & (pval_c < 0.05)
print(f'{sig_c.sum()}/{len(cores)} cores significant at p<0.05')

## 9. Figure 1a — field plus both marker sets

Cyan-edged circles are the stations (MATLAB line 422), yellow-edged diamonds the
ice cores (line 145). No single MATLAB cell draws both — that combination comes
from `figure(3)` being reused across cells.

In [ ]:
fig = plt.figure(figsize=(8, 7.5))
_, ax = polar_axes(fig, 111, lat_max=-62.0)
mesh = ax.contourf(lon_e, lat_e, 10 * slope_e, levels=LEV, cmap=cmap_t,
                   extend='both', transform=PLATE)
add_coast(ax)
scatterm(ax, aws.lat, aws.lon, s=170, c=aws.trend_C_per_dec, cmap=cmap_t,
         vmin=-TREND_LIM, vmax=TREND_LIM, edgecolors=[(0.2, 0.75, 0.75)], linewidths=2.5)
scatterm(ax, cores.lat[sig_c], cores.lon[sig_c], s=190, marker='D',
         c=10 * slope_c[sig_c], cmap=cmap_t, vmin=-TREND_LIM, vmax=TREND_LIM,
         edgecolors=[(0.75, 0.75, 0.2)], linewidths=2.5)
fig.colorbar(mesh, ax=ax, shrink=.65, label='Trend (°C/dec)', ticks=LEV[::2])
ax.set_title(f'Temperature trends ERA5 {YEAR_START}–{YEAR_STOP}')
plt.show()

## 10. Merge ice-core neighbours within 200 km (1.5× scaling)

The grouping is greedy and order-dependent — it walks records in file order and
takes the seed record's coordinates for the whole group. Faithful to the MATLAB.

In [ ]:
g = merge_neighbours(cores.age, cores.d18O, cores.lat, cores.lon, threshold_km=200.0)
sizes = [len(mm) for mm in g['members']]
print(f"{len(cores)} cores → {len(g['age'])} groups "
      f"(largest {max(sizes)}, {sum(1 for s in sizes if s > 1)} actually merged)")

sm = np.array([trend_1d(a, np.asarray(v) * 1.5, CORE_START, CORE_STOP)[:2]
               for a, v in zip(g['age'], g['value'])])
slope_m, pval_m = sm[:, 0], sm[:, 1]
sig_m = np.isfinite(pval_m) & (pval_m < 0.05)
print(f'{sig_m.sum()}/{len(slope_m)} groups p<0.05')

fig = plt.figure(figsize=(8, 7.5))
_, ax = polar_axes(fig, 111, lat_max=-62.0)
mesh = ax.contourf(lon_e, lat_e, 10 * slope_e, levels=LEV, cmap=cmap_t,
                   extend='both', transform=PLATE)
scatterm(ax, g['lat'][sig_m], g['lon'][sig_m], s=200, c=10 * slope_m[sig_m], marker='D',
         cmap=cmap_t, vmin=-TREND_LIM, vmax=TREND_LIM, edgecolors='0.75', linewidths=2)
add_coast(ax)
fig.colorbar(mesh, ax=ax, shrink=.65, label='Trend (°C/dec)', ticks=LEV[::2])
ax.set_title('Merged ice-core trends (1.5×, 200 km groups)')
plt.show()

## 11. Bromwich reconstruction trends

The grid is curvilinear — `lat`/`lon` are 2-D (114×114 WRF polar-stereo), so
`pcolormesh` takes them directly.

In [ ]:
ds_b = load_bromwich()
ann_b = ds_b['RECON'].groupby('time.year').mean('time')
slope_b, pval_b = trend_map(ann_b.year.values.astype(float),
                            ann_b.values.astype('float32'), YEAR_START, YEAR_STOP)
land_b = ds_b['LANDMASK'].values > 0.5

fig = plt.figure(figsize=(8, 7.5))
_, ax = polar_axes(fig, 111, lat_max=-62.0)
mesh = ax.pcolormesh(ds_b['lon'].values, ds_b['lat'].values,
                     10 * np.where(land_b, slope_b, np.nan), cmap=cmap_t,
                     vmin=-TREND_LIM, vmax=TREND_LIM, transform=PLATE, shading='auto')
scatterm(ax, aws.lat, aws.lon, s=170, c=aws.trend_C_per_dec, cmap=cmap_t,
         vmin=-TREND_LIM, vmax=TREND_LIM, edgecolors=[(0.2, 0.75, 0.75)], linewidths=2.5)
add_coast(ax); fig.colorbar(mesh, ax=ax, shrink=.65, label='Trend (°C/dec)')
ax.set_title(f'Bromwich et al. trends {YEAR_START}–{YEAR_STOP}')
plt.show()

## 12. PCA

`pca(data2D)` in the MATLAB treats **grid cells as observations and years as
variables** — the transpose of the usual EOF convention. So `score` reshapes into
spatial maps and `coeff` is the temporal loading. `pca_fields` reproduces that.

In [ ]:
res = pca_fields(cube, n_components=min(80, cube.shape[0]))
exp, cum = res['explained'], res['cumulative']
print(f'PC1 {exp[0]:.1f}%  PC2 {exp[1]:.1f}%  PC3 {exp[2]:.1f}%')
for t in (90, 95, 99):
    k = int(np.searchsorted(cum, t) + 1)
    note = '   <- MATLAB line 502 guesses "20 roughly means 95%"' if t == 95 else ''
    print(f'  {k} PCs reach {t}%{note}')

fig = plt.figure(figsize=(19, 4.4))
for i in range(5):
    _, ax = polar_axes(fig, (1, 5, i + 1), lat_max=-62.0, labels=False)
    v = res['score'][i]
    lim_v = np.nanpercentile(np.abs(v), 99)
    surfm(ax, lat_e, lon_e, v, cmap=cmocean_like('balance', 20), vmin=-lim_v, vmax=lim_v)
    add_coast(ax); ax.set_title(f'PC {i+1} ({exp[i]:.1f}%)')
plt.show()

### Temporal loadings and cumulative variance

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for i in range(3):
    axs[0].plot(years, res['coeff'][:, i], label=f'PC{i+1}')
axs[0].set_xlabel('year'); axs[0].set_ylabel('loading')
axs[0].set_title('Temporal loadings'); axs[0].legend(); axs[0].grid(alpha=.3)
axs[1].plot(np.arange(1, len(cum) + 1), cum, 'o-', ms=3)
axs[1].axhline(95, color='r', ls='--', lw=1)
axs[1].set_xlabel('number of PCs'); axs[1].set_ylabel('cumulative %')
axs[1].set_title('Cumulative variance'); axs[1].grid(alpha=.3)
plt.show()

## 13. Station correlation

Defines `r2` — needed by the next cell. `Stat_data` is rebuilt from the 17 stations
in `dataAWS.mat`; the MATLAB loops to 23, so 6 are not in the delivery.

In [ ]:
sel = (years >= 1950) & (years <= 2021)
yrs = years[sel]

stat = np.full((len(stations), len(yrs)), np.nan)
for i, st in enumerate(stations):
    for y, t in zip(st.year, st.temp):
        j = np.flatnonzero(yrs == y)
        if j.size:
            stat[i, j[0]] = t
print(f'Stat_data: {stat.shape[0]} stations × {stat.shape[1]} years')

coeff = res['coeff'][sel]
corr = np.full((coeff.shape[1], stat.shape[0]), np.nan)
for i in range(coeff.shape[1]):
    for j in range(stat.shape[0]):
        ok = np.isfinite(stat[j]) & np.isfinite(coeff[:, i])
        if ok.sum() > 5:
            corr[i, j] = np.corrcoef(stat[j][ok], coeff[ok, i])[0, 1]
pcid = np.nanmax(np.abs(corr), axis=1) > 0.5
print('PCs correlating |r|>0.5 with a station:', (np.flatnonzero(pcid) + 1).tolist())

anom = cube[sel] - np.nanmean(cube[sel], axis=0, keepdims=True)
T = anom.reshape(len(yrs), -1)
best = np.full(T.shape[1], -np.inf)
for j in range(stat.shape[0]):
    ok = np.isfinite(stat[j])
    if ok.sum() < 10:
        continue
    sv = stat[j][ok] - stat[j][ok].mean()
    Tv = T[ok] - T[ok].mean(axis=0, keepdims=True)
    with np.errstate(invalid='ignore', divide='ignore'):
        r = (sv @ Tv) / np.sqrt((sv @ sv) * (Tv * Tv).sum(axis=0))
    # guard: a constant/all-NaN column gives denom 0, which would leave -inf
    best = np.fmax(best, np.where(np.isfinite(r), r, -np.inf))
r2 = np.where(np.isfinite(best), best ** 2, np.nan).reshape(cube.shape[1:])
print(f'median r² (all cells) = {np.nanmedian(r2):.2f}')

## 14. Figure 1b — variance explained, masked to the continent

In [ ]:
LA_e, LO_e = np.meshgrid(lat_e, lon_e, indexing='ij')
land = elev.land_mask(LA_e, LO_e)          # BedMachine mask, not an elevation cut

fig = plt.figure(figsize=(8, 7.5))
_, ax = polar_axes(fig, 111, lat_max=-62.0)
mesh = ax.contourf(lon_e, lat_e, np.where(land, r2, np.nan),
                   levels=np.arange(0, 1.01, 0.1),
                   cmap=cmocean_like('rain', 10), transform=PLATE)
add_coast(ax)
scatterm(ax, cores.lat, cores.lon, s=55, marker='s',
         c=[(212/255, 23/255, 35/255)], edgecolors='k', linewidths=.6)
fig.colorbar(mesh, ax=ax, shrink=.65, label='r$^2$', ticks=np.arange(0, 1.01, .1))
ax.set_title('Amount of variance explained')
plt.show()

r2l = np.where(land, r2, np.nan)
print(f'median r² over land = {np.nanmedian(r2l):.2f}  '
      f'(range {np.nanmin(r2l):.2f}–{np.nanmax(r2l):.2f})')

## 15. New team ice cores — not in the delivery

In [ ]:
from antarctic.datasets import load_paleo_ldc
try:
    dataPALEO, dataLDC = load_paleo_ldc()
    print('found it — extend this cell')
except FileNotFoundError as e:
    print('SKIPPED:', e)